In [1]:
import Pkg; Pkg.add("CSV")
import Pkg; Pkg.add("DataFrames")
import Pkg; Pkg.add("Statistics")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
    Updating `~/.julia/environments/v1.11/Project.toml`
  [10745b16] + Statistics v1.11.5
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [ ]:
using CSV, DataFrames, Statistics

nonpoint = DataFrame(CSV.File(expanduser("fertManure.csv")))
point = DataFrame(CSV.File(expanduser("ptSource_WWTP_CSO.csv")))

# RIM stations
stations = ["01491000", "01578310", "01594440", 
            "01646580", "01668000", "01673000", 
            "01674500", "02035000", "02041650"]

# format station IDs
nonpoint.staid = lpad.(string.(nonpoint.staid), 8, '0')
point.staid = lpad.(string.(point.staid), 8, '0')

# filter for nine stations
nonpoint_df = filter(row -> row.staid in stations, nonpoint)
point_df = filter(row -> row.staid in stations, point)

# select columns
nonpoint_df = select(nonpoint_df, :staid, :year, :fertTN, :manureTN) # lbs N
point_df = select(point_df, :staid, :year, :ptsourcetn, :csotn) # lbs N

# calculate total nonpoint nitrogen
nonpoint_df.nonpoint_TN = nonpoint_df.fertTN .+ nonpoint_df.manureTN

# calculate total point nitrogen ASSUMING cso included
point_df.point_TN = point_df.ptsourcetn .+ point_df.csotn

# sort by station and year
sort!(nonpoint_df, [:staid, :year])
sort!(point_df, [:staid, :year])

# aggregate nonpoint TN by station (avg across years)
nonpoint_agg = combine(groupby(nonpoint_df, :staid),:nonpoint_TN => mean => :nonpoint_TN)

# aggregate point TN by station (avg across years)
point_agg = combine(groupby(point_df, :staid),:point_TN => mean => :point_TN)

# sort by station ID
sort!(nonpoint_agg, :staid)
sort!(point_agg, :staid)

#println(nonpoint_agg)
#println(point_agg)

# aggregate flow
flow = DataFrame(CSV.File("RIM_2023_AnnualLoadTable.csv"))
flow.STAID = lpad.(string.(flow.STAID), 8, '0')
flow_df = filter(row -> row.STAID in stations, flow)

# calculate average Q for each station
flow_agg = combine(groupby(flow_df, :STAID),:Q => mean => :Q) # lb/ft3

sort!(flow_agg, :STAID)
#println(flow_agg)

# available area for buffer for each staid
buffer = DataFrame(CSV.File("Buffer.csv"))
buffer.staid = lpad.(string.(buffer.staid), 8, '0')
get_A(station) = filter(row -> row.staid == station, buffer).available_area[1] # acres

# example use in optimization to retrieve available area for buffer for staid
# A_01491000 = get_A("01491000")

3616